# CS 4412 — M2: Initial Implementation (Discovery-Oriented)
**Project:** Power Naps vs. Coffee (Fatigue-Management Patterns)  
**Author:** Kofi Ofori-Acquah  
**Dataset:** `power_nap_vs_coffee_effectiveness_dataset.csv` (500 × 11)

**Goal of this notebook:**  
1) Understand the data (EDA) → 2) Prepare it (clean + engineer) → 3) Apply **one mining technique** → 4) Interpret at least one discovered pattern.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Display settings (optional)
pd.set_option("display.max_columns", 50)


In [ ]:
# 1) Load data
DATA_PATH = r"/mnt/data/power_nap_vs_coffee_effectiveness_dataset.csv"
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
df_raw.head()


## 2) Data overview (types, missingness, duplicates)
We quickly profile the dataset to understand what we have before mining.


In [ ]:
df = df_raw.copy()

# Basic info
display(df.dtypes)
print("\nMissing values per column:")
display(df.isna().sum())

# Duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")


## 3) Cleaning + feature engineering
**Decisions (documented):**
- `side_effects`: missing values become **"None/Not Reported"** (keeps rows; missingness itself is informative).
- Remove exact duplicate rows.
- Engineer **alertness_delta = alertness_score_after − alertness_score_before** (key discovery feature).


In [ ]:
# Fill missing side_effects
if "side_effects" in df.columns:
    df["side_effects"] = df["side_effects"].fillna("None/Not Reported")

# Drop exact duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Feature engineering
df["alertness_delta"] = df["alertness_score_after"] - df["alertness_score_before"]

df.head()


## 4) EDA (purposeful, tied to discovery questions)
We want to understand:
- Distributions (sleep, durations, outcomes)
- Relationships (sleep/duration vs delta)
- Differences by **intervention type** (coffee vs nap)
- Any suspicious values (outliers/extremes)

Below are **7** visualizations + short reads.


In [ ]:
# Helper: simple plotting wrapper
def show():
    plt.tight_layout()
    plt.show()

# 1) Distribution: sleep hours
plt.figure()
plt.hist(df["sleep_hours_previous_night"], bins=15)
plt.title("Sleep hours previous night (distribution)")
plt.xlabel("Hours"); plt.ylabel("Count")
show()

# 2) Distribution: intervention duration
plt.figure()
plt.hist(df["intervention_duration_minutes"], bins=15)
plt.title("Intervention duration (minutes)")
plt.xlabel("Minutes"); plt.ylabel("Count")
show()

# 3) Distribution: alertness_delta
plt.figure()
plt.hist(df["alertness_delta"], bins=15)
plt.title("Alertness delta (after - before)")
plt.xlabel("Delta"); plt.ylabel("Count")
show()

# 4) Boxplot: delta by intervention type
plt.figure()
types = df["intervention_type"].unique()
data = [df.loc[df["intervention_type"]==t, "alertness_delta"] for t in types]
plt.boxplot(data, labels=types)
plt.title("Alertness delta by intervention type")
plt.ylabel("Alertness delta")
show()

# 5) Scatter: sleep vs delta (colored by intervention type)
plt.figure()
for t in types:
    sub = df[df["intervention_type"]==t]
    plt.scatter(sub["sleep_hours_previous_night"], sub["alertness_delta"], s=15, alpha=0.7, label=t)
plt.title("Sleep vs Alertness delta (by intervention type)")
plt.xlabel("Sleep hours previous night"); plt.ylabel("Alertness delta")
plt.legend()
show()

# 6) Scatter: duration vs delta (colored by intervention type)
plt.figure()
for t in types:
    sub = df[df["intervention_type"]==t]
    plt.scatter(sub["intervention_duration_minutes"], sub["alertness_delta"], s=15, alpha=0.7, label=t)
plt.title("Duration vs Alertness delta (by intervention type)")
plt.xlabel("Intervention duration (min)"); plt.ylabel("Alertness delta")
plt.legend()
show()

# 7) Correlation heatmap (numeric only)
numeric_cols = [
    "age",
    "sleep_hours_previous_night",
    "intervention_duration_minutes",
    "alertness_score_before",
    "alertness_score_after",
    "alertness_delta",
    "productivity_rating",
    "mood_rating",
]
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(7,6))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation heatmap (numeric features)")
plt.colorbar()
show()

# Quick grouped summaries (simple tables)
display(df.groupby("intervention_type")[["alertness_delta","productivity_rating","mood_rating"]].mean())


## 5) Data transformation for clustering
Clustering is distance-based, so **scale matters**.  
We standardize numeric features using **z-score** so no single feature dominates distance.


In [ ]:
# Choose numeric features for clustering (simple, outcome-focused)
features = [
    "sleep_hours_previous_night",
    "intervention_duration_minutes",
    "alertness_score_before",
    "alertness_score_after",
    "alertness_delta",
    "productivity_rating",
    "mood_rating",
]

X = df[features].to_numpy(dtype=float)

scaler = StandardScaler()
Z = scaler.fit_transform(X)

print("Z shape:", Z.shape)


## 6) Mining technique: K-Means clustering
We apply **K-Means** to discover natural segments of sessions based on:
sleep, duration, alertness before/after, delta, productivity, mood.

**Choosing k (simple approach):**
We compute silhouette scores for k=2..8 and pick the best score.


In [ ]:
k_range = range(2, 9)
scores = []
models = []

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(Z)
    s = silhouette_score(Z, labels)
    scores.append(s)
    models.append(km)

best_idx = int(np.argmax(scores))
best_k = list(k_range)[best_idx]
best_score = scores[best_idx]

print("Silhouette scores:")
for k, s in zip(k_range, scores):
    print(f"k={k}: {s:.4f}")
print(f"\nBest k = {best_k} (silhouette={best_score:.4f})")


In [ ]:
# Fit final K-Means
km = models[best_idx]
df["cluster"] = km.labels_

df[["cluster"]].value_counts().sort_index()


## 7) Cluster profiles (what makes clusters distinct?)
We summarize each cluster by mean feature values and also check composition by intervention type / occupation.


In [ ]:
cluster_means = df.groupby("cluster")[features].mean(numeric_only=True)
display(cluster_means)

# Categorical composition
for col in ["intervention_type", "occupation"]:
    print(f"\n{col} proportion by cluster:")
    comp = (
        df.groupby("cluster")[col]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
        .sort_values(["cluster","proportion"], ascending=[True, False])
    )
    display(comp)


## 8) Simple visualization of clusters (PCA 2D)
PCA is used **only for visualization** (not for the clustering itself).


In [ ]:
pca = PCA(n_components=2, random_state=42)
Z2 = pca.fit_transform(Z)

plt.figure()
for c in sorted(df["cluster"].unique()):
    sub = Z2[df["cluster"].values == c]
    plt.scatter(sub[:,0], sub[:,1], s=15, alpha=0.7, label=f"Cluster {c}")
plt.title(f"PCA view of clusters (k={best_k})")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()


## 9) Preliminary findings (the “So what?”)
Use the cluster profiles above to write **human-readable** segments.  
Example template:
- **Cluster A:** “Low sleep + big delta + high productivity” → suggests *X works best under Y conditions* in this dataset.

Below is a simple auto-generated *starter* description you can refine.


In [ ]:
# Starter: name clusters by relative patterns (simple heuristics)
means = cluster_means.copy()

# For easier reading, rank clusters by alertness_delta mean
ranked = means.sort_values("alertness_delta", ascending=False)
display(ranked[["sleep_hours_previous_night","intervention_duration_minutes","alertness_delta","productivity_rating","mood_rating"]])

print("\nStarter interpretations (edit these):")
for cluster_id, row in ranked.iterrows():
    sleep = row["sleep_hours_previous_night"]
    dur = row["intervention_duration_minutes"]
    delta = row["alertness_delta"]
    prod = row["productivity_rating"]
    mood = row["mood_rating"]
    print(f"- Cluster {cluster_id}: sleep≈{sleep:.2f}h, duration≈{dur:.1f}m, delta≈{delta:.2f}, prod≈{prod:.2f}, mood≈{mood:.2f}")


## 10) Export (optional)
If you want a CSV with clusters (for your repo), export it here.


In [ ]:
OUT_PATH = "m2_clustered_output.csv"
df.to_csv(OUT_PATH, index=False)
print("Wrote:", OUT_PATH)
